# Fink LSST — TNS Supernovae Light Curves (`fink_in_tns_lsst`)

This notebook retrieves alerts tagged as TNS supernovae by the Fink broker via the new tag
`fink_in_tns_lsst`, downloads their multi-band light curves, and displays them per object.

**Tag description**: Objects identified in the Transient Name Server (TNS) with a confirmed or
candidate supernova classification, cross-matched by the Fink broker against the LSST alert stream.
This tag supersedes the older `in_tns` tag and is specific to the LSST data flow.

**API reference**: https://api.lsst.fink-portal.org  
**Schema browser**: https://lsst.fink-portal.org/schemas

**Column naming convention (LSST DPDD)**
- `r:<col>` — original LSST diaSource/diaObject field. The `r:` prefix is the **table name**, NOT the r spectral band.
- `f:<col>` — Fink-computed field (classifiers, cross-match results).
- Spectral band → value of `r:band` ∈ {`u`, `g`, `r`, `i`, `z`, `y`}.

**Redshift availability**: The redshift of the host galaxy or the SN itself may be available from:
- `f:xm_tns_redshift`  — spectroscopic redshift reported in TNS (most reliable; may be NaN if not yet reported)
- `f:xm_legacydr8_zphot` — photometric redshift from Legacy Survey DR8 (host galaxy proxy)
- `f:xm_mangrove_lum_dist` — luminosity distance from the MANGROVE galaxy catalogue

Note: `f:xm_tns_redshift` is the new column added with the `in_tns` tag.
It will be `NaN` for objects where TNS has not yet received a spectroscopic classification.

---

**Notebook structure**
1. Imports and configuration
2. Download catalog from Fink API (`/api/v1/tags` with `in_tns`)
3. Explore catalog: TNS types, redshift distribution, sky map
4. Download light curves (`/api/v1/sources`)
5. Multi-band light curve plots per object (flux and AB magnitude)
6. Summary table with TNS name, type, redshift, and light curve statistics
7. Save results to disk (parquet + figures)


## 1 — Imports and configuration

In [ ]:
import io
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import requests
from astropy.table import Table
from astropy.visualization import ZScaleInterval

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Imports OK")

- check colors here : https://www.w3schools.com/colors/colors_picker.asp

In [ ]:
# ── Fink LSST API ─────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org/api/v1"

# ── Tag to query ───────────────────────────────────────────────────────────────
# New tag specific to LSST TNS supernovae (introduced 2026)
TAG = "in_tns"

# ── Number of alerts to fetch ─────────────────────────────────────────────────
N_ALERTS = 1000  # increase as needed; unique diaObjects after deduplication will be fewer

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "NB07_TNS_SN"
DATA_DIR = Path(f"data_{NB_TAG}")
FIGS_DIR = Path(f"figs_{NB_TAG}")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Rate-limiting: pause between API calls ────────────────────────────────────
API_DELAY = 0.4  # seconds

# ── Rubin AB zeropoint (nJy) ──────────────────────────────────────────────────
RUBIN_ZEROPOINT = 31.4  # AB mag such that flux 1 nJy → mag RUBIN_ZEROPOINT

# ── Band colours ──────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9B59B6",  # violet
    "g": "#2ECC71",  # green
    "r": "#C0392B",  # red
    "i": "#C0392B",  # light red
    "z": "#C0392B",  # orange
    "y": "#C0392B",  # brown
}

# ── Dark theme ────────────────────────────────────────────────────────────────
DARK_BG = "#0D1117"
PANEL_BG = "#161B22"
TEXT_COL = "#E6EDF3"
MUTED_COL = "#8B949E"
ACCENT = "#58A6FF"
HIGHLIGHT = "#FF7B72"

print(f"Tag          : {TAG}")
print(f"N alerts     : {N_ALERTS}")
print(f"Data dir     : {DATA_DIR.resolve()}")
print(f"Figures dir  : {FIGS_DIR.resolve()}")

## 2 — Download catalog from Fink API

We query the `/api/v1/tags` endpoint for the `fink_in_tns_lsst` tag.
The key TNS cross-match columns are:
- `f:xm_tns_fullname` — TNS internal name (e.g. SN 2026abc)
- `f:xm_tns_type`     — object type reported to TNS (SN Ia, SN II, …)
- `f:xm_tns_redshift` — spectroscopic redshift from TNS (new with `fink_in_tns_lsst`)

In [ ]:
# Columns to fetch from the catalog endpoint
CATALOG_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:band",
        "r:visit",
        "r:detector",
        "r:x",
        "r:y",
        "r:target_name",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:scienceFlux",
        "r:scienceFluxErr",
        "r:apFlux",
        "r:apFluxErr",
        "r:templateFlux",
        "r:templateFluxErr",
        "r:snr",
        "r:reliability",
        "r:extendedness",
        "r:isNegative",
        "r:isDipole",
        "r:dipoleFluxDiff",
        "r:dipoleFluxDiffErr",
        "r:dipoleMeanFlux",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleMeanFluxErr",
        "r:dipoleNdata",
        "r:dipoleChi2",
        "r:dipoleFitAttempted",
        # Fink SN classifiers
        "f:clf_cats_class",
        "f:clf_cats_score",
        "f:clf_snnSnVsOthers_score",
        "f:clf_earlySNIa_score",
        # TNS cross-match  ← key columns for this notebook
        "f:xm_tns_fullname",
        "f:xm_tns_type",
        "f:xm_tns_redshift",  # spectroscopic z from TNS (may be NaN)
        # Other cross-matches (host galaxy proxies)
        "f:xm_simbad_otype",
        "f:xm_legacydr8_zphot",  # photometric z from Legacy Survey DR8
        "f:xm_legacydr8_e_zphot",
        "f:xm_legacydr8_fqual",
        "f:xm_legacydr8_pstar",
        "f:xm_mangrove_lum_dist",
        "f:xm_gaiadr3_DR3Name",
        "f:xm_gaiadr3_VarFlag",
        "f:xm_gaiadr3_Plx",
        "f:xm_gaiadr3_e_Plx",
        "f:xm_gcvs_type",
        "f:xm_mangrove_2MASS_name",
        "f:xm_mangrove_HyperLEDA_name",
        "f:xm_mangrove_ang_dist",
        "f:xm_mangrove_lum_dist",
        "f:xm_vsx_type",
        "f:xm_x3hsp_type",
        "f:xm_x4lac_type",
    ]
)

LC_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:band",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:snr",
        "r:reliability",
    ]
)


def fetch_catalog(tag: str, n: int) -> pd.DataFrame:
    """Query /api/v1/tags for a given Fink LSST tag."""
    params = {
        "tag": tag,
        "n": n,
        "columns": CATALOG_COLUMNS,
        "output-format": "json",
    }
    print(f"Querying {FINK_API}/tags  tag={tag}  n={n} ...")
    resp = requests.get(f"{FINK_API}/tags", params=params, timeout=120)
    resp.raise_for_status()
    if not resp.text.strip():
        print("[warn] Empty response from API — tag may have no data yet.")
        return pd.DataFrame()
    df = pd.read_json(io.BytesIO(resp.content))
    print(f"  -> {len(df)} alerts received.")
    return df


raw_catalog = fetch_catalog(TAG, N_ALERTS)

In [ ]:
# Fallback: if fink_in_tns_lsst is not yet populated, fall back to in_tns
if raw_catalog.empty:
    FALLBACK_TAG = "in_tns"
    print(f"[info] fink_in_tns_lsst returned no data; falling back to tag '{FALLBACK_TAG}'.")
    raw_catalog = fetch_catalog(FALLBACK_TAG, N_ALERTS)

print(f"Raw catalog shape: {raw_catalog.shape}")
print("Columns:", raw_catalog.columns.tolist())

In [ ]:
# Deduplicate: keep one row per diaObjectId (most recent midpointMjdTai)
if "r:midpointMjdTai" in raw_catalog.columns:
    catalog = (
        raw_catalog.sort_values("r:midpointMjdTai")
        .drop_duplicates(subset="r:diaObjectId", keep="last")
        .reset_index(drop=True)
    )
else:
    catalog = raw_catalog.drop_duplicates(subset="r:diaObjectId").reset_index(drop=True)

print(f"Unique diaObjects after deduplication: {len(catalog)}")
catalog.head(10)

In [ ]:
# Save catalog to disk
cat_path = DATA_DIR / "catalog_fink_in_tns_lsst.parquet"
catalog.to_parquet(cat_path, index=False)
print(f"Catalog saved -> {cat_path}")

## 3 — Explore catalog

### 3.1 TNS type distribution

In [ ]:
tns_type_col = "f:xm_tns_type"
tns_name_col = "f:xm_tns_fullname"
tns_z_col = "f:xm_tns_redshift"
zphot_col = "f:xm_legacydr8_zphot"

if tns_type_col in catalog.columns:
    type_counts = catalog[tns_type_col].fillna("unknown").value_counts()
    print("TNS type distribution:")
    print(type_counts.to_string())
else:
    print(f"[warn] Column {tns_type_col!r} not present in catalog.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=DARK_BG)

# --- TNS type bar chart ---
ax = axes[0]
ax.set_facecolor(PANEL_BG)
if tns_type_col in catalog.columns:
    type_counts = catalog[tns_type_col].fillna("unknown").value_counts()
    bars = ax.barh(type_counts.index[::-1], type_counts.values[::-1], color=ACCENT, edgecolor=DARK_BG)
    ax.set_xlabel("Number of objects", color=MUTED_COL)
    ax.set_title("TNS type distribution", color=ACCENT)
    ax.tick_params(colors=MUTED_COL, labelsize=8)
else:
    ax.text(0.5, 0.5, "No TNS type column", color=MUTED_COL, ha="center", va="center", transform=ax.transAxes)

# --- Redshift distribution ---
ax = axes[1]
ax.set_facecolor(PANEL_BG)
z_vals_tns = pd.to_numeric(catalog.get(tns_z_col, pd.Series(dtype=float)), errors="coerce").dropna()
z_vals_phot = pd.to_numeric(catalog.get(zphot_col, pd.Series(dtype=float)), errors="coerce").dropna()

if len(z_vals_tns) > 0:
    ax.hist(
        z_vals_tns,
        bins=30,
        color=HIGHLIGHT,
        alpha=0.8,
        label=f"z_TNS (spec) N={len(z_vals_tns)}",
        density=False,
    )
if len(z_vals_phot) > 0:
    ax.hist(
        z_vals_phot,
        bins=30,
        color=ACCENT,
        alpha=0.5,
        label=f"z_LegacyDR8 (phot) N={len(z_vals_phot)}",
        density=False,
    )
if len(z_vals_tns) == 0 and len(z_vals_phot) == 0:
    ax.text(0.5, 0.5, "No redshift data", color=MUTED_COL, ha="center", va="center", transform=ax.transAxes)

ax.set_xlabel("Redshift z", color=MUTED_COL)
ax.set_ylabel("Count", color=MUTED_COL)
ax.set_title("Redshift distribution", color=ACCENT)
ax.tick_params(colors=MUTED_COL)
ax.legend(fontsize=8, facecolor=DARK_BG, labelcolor=TEXT_COL)
ax.set_facecolor(PANEL_BG)

plt.suptitle(f"Fink LSST — {TAG} — N={len(catalog)} objects", color=TEXT_COL, fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(FIGS_DIR / "catalog_overview.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()
print(f"  Spec-z from TNS  : {len(z_vals_tns)} objects have a value")
print(f"  Photo-z LegacyDR8: {len(z_vals_phot)} objects have a value")

### 3.2 Sky distribution (RA/Dec)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)

z_color = pd.to_numeric(catalog.get(tns_z_col, pd.Series(dtype=float)), errors="coerce")
has_z = z_color.notna()

# Points without spec-z in grey
ax.scatter(
    catalog.loc[~has_z, "r:ra"],
    catalog.loc[~has_z, "r:dec"],
    color=MUTED_COL,
    s=12,
    alpha=0.5,
    label="no spec-z",
)
# Points with spec-z coloured by z value
if has_z.any():
    sc = ax.scatter(
        catalog.loc[has_z, "r:ra"],
        catalog.loc[has_z, "r:dec"],
        c=z_color[has_z],
        cmap="plasma",
        s=30,
        alpha=0.9,
        vmin=0,
        vmax=z_color[has_z].quantile(0.95) if has_z.sum() > 5 else 1.0,
        label="spec-z from TNS",
    )
    plt.colorbar(sc, ax=ax, label="spec-z (TNS)")

ax.set_xlabel("RA (deg)", color=MUTED_COL)
ax.set_ylabel("Dec (deg)", color=MUTED_COL)
ax.set_title(f"{TAG} — sky distribution  (N={len(catalog)})", color=ACCENT)
ax.tick_params(colors=MUTED_COL)
ax.legend(fontsize=8, facecolor=DARK_BG, labelcolor=TEXT_COL)
plt.tight_layout()
fig.savefig(FIGS_DIR / "sky_distribution.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()

### 3.3 Summary table: TNS objects with redshift

In [ ]:
display_cols = [
    "r:diaObjectId",
    "r:ra",
    "r:dec",
    "r:midpointMjdTai",
    "f:xm_tns_fullname",
    "f:xm_tns_type",
    "f:xm_tns_redshift",
    "f:xm_legacydr8_zphot",
    "f:clf_snnSnVsOthers_score",
    "f:clf_earlySNIa_score",
]
avail_cols = [c for c in display_cols if c in catalog.columns]
summary_tbl = catalog[avail_cols].copy()

# Sort by spec-z if available, else by MJD
if tns_z_col in summary_tbl.columns:
    summary_tbl = summary_tbl.sort_values(tns_z_col)
elif "r:midpointMjdTai" in summary_tbl.columns:
    summary_tbl = summary_tbl.sort_values("r:midpointMjdTai", ascending=False)

print(f"{len(summary_tbl)} objects.  Columns: {avail_cols}")
summary_tbl.reset_index(drop=True)

## 4 — Download light curves

We fetch per-diaObject light curves from `/api/v1/sources`.
All bands (`u`, `g`, `r`, `i`, `z`, `y`) are retrieved.

In [ ]:
LC_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:band",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:snr",
        "r:reliability",
    ]
)

LC_DIR = DATA_DIR / "light_curves"
LC_DIR.mkdir(parents=True, exist_ok=True)


def fetch_lightcurve(obj_id: int) -> pd.DataFrame:
    """Fetch multi-band light curve for one diaObjectId from Fink LSST API."""
    params = {
        "diaObjectId": obj_id,
        "columns": LC_COLUMNS,
        "output-format": "json",
    }
    resp = requests.get(f"{FINK_API}/sources", params=params, timeout=60)
    resp.raise_for_status()
    if not resp.text.strip():
        return pd.DataFrame()
    try:
        df = pd.read_json(io.BytesIO(resp.content))
    except Exception:
        return pd.DataFrame()
    if "r:midpointMjdTai" in df.columns:
        df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)
    return df


print(f"Will download light curves for {len(catalog)} objects into {LC_DIR}.")

In [ ]:
lc_cache: dict[int, pd.DataFrame] = {}
n_ok, n_fail = 0, 0

for i, row in catalog.iterrows():
    obj_id = int(row["r:diaObjectId"])
    lc_path = LC_DIR / f"lc_{obj_id}.parquet"

    if lc_path.exists():
        lc = pd.read_parquet(lc_path)
        lc_cache[obj_id] = lc
        n_ok += 1
        continue

    try:
        lc = fetch_lightcurve(obj_id)
        if not lc.empty:
            lc.to_parquet(lc_path, index=False)
            lc_cache[obj_id] = lc
            n_ok += 1
        else:
            n_fail += 1
    except Exception as exc:
        print(f"  [warn] diaObjectId={obj_id}: {exc}")
        n_fail += 1

    time.sleep(API_DELAY)

    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(catalog)} done  (ok={n_ok}, fail={n_fail})")

print(f"\nLight curve download complete: ok={n_ok}, fail={n_fail}")

## 5 — Multi-band light curve plots

### 5.1 Helper: plot one object

For each object we show:
- Top panel: PSF flux (nJy) vs MJD, one colour per band
- Bottom panel: AB magnitude vs MJD (detected points only, SNR > 3)

In [ ]:
def flux_to_mag(flux_nJy: np.ndarray) -> np.ndarray:
    """Convert PSF flux in nJy to AB magnitude (Rubin convention)."""
    with np.errstate(divide="ignore", invalid="ignore"):
        mag = -2.5 * np.log10(np.where(flux_nJy > 0, flux_nJy, np.nan)) + RUBIN_ZEROPOINT
    return mag


def plot_tns_lightcurve(
    lc: pd.DataFrame,
    meta: pd.Series,
    save_path: Path | None = None,
) -> None:
    """
    Plot multi-band flux and magnitude light curve for one TNS SN.

    Parameters
    ----------
    lc   : DataFrame from /api/v1/sources
    meta : catalog row for this object
    save_path : if given, save figure to this path
    """
    obj_id = int(meta["r:diaObjectId"])
    tns_name = meta.get("f:xm_tns_fullname", "—")
    tns_type = meta.get("f:xm_tns_type", "—")
    z_spec = meta.get("f:xm_tns_redshift", np.nan)
    z_phot = meta.get("f:xm_legacydr8_zphot", np.nan)
    snn_score = meta.get("f:clf_snnSnVsOthers_score", np.nan)
    snia_score = meta.get("f:clf_earlySNIa_score", np.nan)

    try:
        z_spec = float(z_spec)
    except (TypeError, ValueError):
        z_spec = np.nan
    try:
        z_phot = float(z_phot)
    except (TypeError, ValueError):
        z_phot = np.nan

    if lc.empty:
        print(f"  [skip] diaObjectId={obj_id} — empty light curve.")
        return

    bands_present = sorted(lc["r:band"].unique())
    mjd = lc["r:midpointMjdTai"].values
    mjd0 = mjd.min()
    delta_t = mjd - mjd0

    fig = plt.figure(figsize=(10, 6), facecolor=DARK_BG)
    gs = gridspec.GridSpec(2, 1, hspace=0.05, figure=fig)
    ax_flux = fig.add_subplot(gs[0])
    ax_mag = fig.add_subplot(gs[1], sharex=ax_flux)

    for ax in (ax_flux, ax_mag):
        ax.set_facecolor(PANEL_BG)
        ax.tick_params(colors=MUTED_COL)
        for spine in ax.spines.values():
            spine.set_edgecolor(MUTED_COL)

    for band in bands_present:
        mask = lc["r:band"] == band
        dt_b = delta_t[mask]
        flux_b = lc.loc[mask, "r:psfFlux"].values
        ferr_b = lc.loc[mask, "r:psfFluxErr"].values
        snr_b = lc.loc[mask, "r:snr"].values if "r:snr" in lc.columns else np.ones(mask.sum()) * 5
        col = BAND_COLORS.get(band, "white")

        # Flux panel — all points (detections and non-detections)
        det = snr_b >= 3
        ax_flux.errorbar(
            dt_b[det],
            flux_b[det],
            yerr=ferr_b[det],
            fmt="o",
            color=col,
            ms=4,
            capsize=2,
            lw=0.8,
            label=f"{band} ({det.sum()})",
        )
        # Upper limits (SNR < 3)
        if (~det).any():
            ax_flux.scatter(dt_b[~det], flux_b[~det], marker="v", color=col, s=20, alpha=0.4)

        # Magnitude panel — detections only
        mag_b = flux_to_mag(flux_b[det])
        valid_mag = np.isfinite(mag_b)
        if valid_mag.any():
            # Magnitude error via error propagation
            mag_err = 2.5 / np.log(10) * ferr_b[det][valid_mag] / flux_b[det][valid_mag]
            ax_mag.errorbar(
                dt_b[det][valid_mag],
                mag_b[valid_mag],
                yerr=mag_err,
                fmt="o",
                color=col,
                ms=4,
                capsize=2,
                lw=0.8,
            )

    # Axis labels
    ax_flux.set_ylabel("PSF Flux (nJy)", color=MUTED_COL)
    ax_mag.set_ylabel("AB magnitude", color=MUTED_COL)
    ax_mag.set_xlabel(f"Days since MJD {mjd0:.1f}", color=MUTED_COL)
    ax_mag.invert_yaxis()
    plt.setp(ax_flux.get_xticklabels(), visible=False)

    # Legend on flux panel
    ax_flux.legend(
        title="band (Ndet)",
        fontsize=8,
        title_fontsize=8,
        facecolor=DARK_BG,
        labelcolor=TEXT_COL,
        loc="upper right",
    )

    # Build title with redshift info
    z_str_parts = []
    if np.isfinite(z_spec):
        z_str_parts.append(f"z_spec={z_spec:.4f} (TNS)")
    if np.isfinite(z_phot):
        z_str_parts.append(f"z_phot={z_phot:.3f} (LegacyDR8)")
    z_str = "  |  ".join(z_str_parts) if z_str_parts else "redshift: n/a"

    score_str = ""
    try:
        score_str = f"SNN={float(snn_score):.3f}  earlySNIa={float(snia_score):.3f}"
    except (TypeError, ValueError):
        pass

    title_line1 = f"diaObjectId {obj_id}  |  {tns_name}  |  type: {tns_type}"
    title_line2 = f"{z_str}  |  RA={meta['r:ra']:.4f}  Dec={meta['r:dec']:.4f}"
    title_line3 = score_str

    fig.suptitle(
        "\n".join(filter(None, [title_line1, title_line2, title_line3])),
        color=TEXT_COL,
        fontsize=9,
        y=1.01,
    )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=DARK_BG)

    plt.show()
    plt.close(fig)


print("plot_tns_lightcurve() defined.")

### 5.2 Plot light curves — objects with known TNS spec-z first

We sort by availability of spectroscopic redshift from TNS, then plot a configurable number of objects.

In [ ]:
# ── USER PARAMETER ────────────────────────────────────────────────────────────
MAX_OBJECTS_TO_PLOT = 50  # set to None to plot all
SAVE_FIGURES = True  # set to False to skip saving
# ─────────────────────────────────────────────────────────────────────────────

# Sort: objects with TNS spec-z first
if tns_z_col in catalog.columns:
    z_numeric = pd.to_numeric(catalog[tns_z_col], errors="coerce")
    sort_idx = np.argsort(z_numeric.isna().values.astype(int))  # non-NaN first
    sorted_catalog = catalog.iloc[sort_idx].reset_index(drop=True)
else:
    sorted_catalog = catalog.copy()

obj_ids_to_plot = sorted_catalog["r:diaObjectId"].tolist()
if MAX_OBJECTS_TO_PLOT is not None:
    obj_ids_to_plot = obj_ids_to_plot[:MAX_OBJECTS_TO_PLOT]

print(f"Plotting {len(obj_ids_to_plot)} light curves ...")

for oid in obj_ids_to_plot:
    oid = int(oid)
    lc = lc_cache.get(oid)
    if lc is None:
        lc_path = LC_DIR / f"lc_{oid}.parquet"
        if lc_path.exists():
            lc = pd.read_parquet(lc_path)
        else:
            print(f"  [skip] No light curve for diaObjectId={oid}")
            continue

    meta_rows = sorted_catalog[sorted_catalog["r:diaObjectId"] == oid]
    if meta_rows.empty:
        continue
    meta = meta_rows.iloc[0]

    tns_name_safe = str(meta.get("f:xm_tns_fullname", oid)).replace(" ", "_")
    fig_path = (FIGS_DIR / f"lc_{oid}_{tns_name_safe}.png") if SAVE_FIGURES else None

    plot_tns_lightcurve(lc, meta, save_path=fig_path)

## 6 — Summary table: light curve statistics per object

In [ ]:
rows = []
for _, meta in catalog.iterrows():
    oid = int(meta["r:diaObjectId"])
    lc = lc_cache.get(oid)
    if lc is None:
        lc_path = LC_DIR / f"lc_{oid}.parquet"
        lc = pd.read_parquet(lc_path) if lc_path.exists() else pd.DataFrame()

    n_det = len(lc)
    bands = sorted(lc["r:band"].unique().tolist()) if not lc.empty else []
    mjd_first = lc["r:midpointMjdTai"].min() if not lc.empty else np.nan
    mjd_last = lc["r:midpointMjdTai"].max() if not lc.empty else np.nan

    try:
        z_spec = float(meta.get(tns_z_col, np.nan))
    except (TypeError, ValueError):
        z_spec = np.nan
    try:
        z_phot = float(meta.get(zphot_col, np.nan))
    except (TypeError, ValueError):
        z_phot = np.nan

    rows.append(
        {
            "diaObjectId": oid,
            "TNS_name": meta.get("f:xm_tns_fullname", ""),
            "TNS_type": meta.get("f:xm_tns_type", ""),
            "z_spec_TNS": z_spec,
            "z_phot_DR8": z_phot,
            "RA": meta["r:ra"],
            "Dec": meta["r:dec"],
            "N_detections": n_det,
            "bands": ",".join(bands),
            "MJD_first": mjd_first,
            "MJD_last": mjd_last,
            "duration_days": (mjd_last - mjd_first) if (not np.isnan(mjd_first) and n_det > 1) else 0.0,
            "SNN_score": meta.get("f:clf_snnSnVsOthers_score", np.nan),
            "earlySNIa_score": meta.get("f:clf_earlySNIa_score", np.nan),
        }
    )

summary_df = pd.DataFrame(rows)
# Sort: spec-z available first, then by TNS name
summary_df = summary_df.sort_values(["z_spec_TNS", "TNS_name"], na_position="last").reset_index(drop=True)

out_csv = DATA_DIR / "summary_tns_sn.csv"
summary_df.to_csv(out_csv, index=False)
print(f"Summary table saved -> {out_csv}")
summary_df

## 7 — Diagnostics: redshift vs light curve duration

For objects with a known spec-z, we can check whether the observed light curve duration
is consistent with expectations (time dilation: Δt_obs ≈ Δt_rest × (1+z)).

In [ ]:
mask_z = summary_df["z_spec_TNS"].notna() & (summary_df["duration_days"] > 0)

if mask_z.sum() >= 2:
    fig, ax = plt.subplots(figsize=(7, 4), facecolor=DARK_BG)
    ax.set_facecolor(PANEL_BG)

    sub = summary_df[mask_z]
    sc = ax.scatter(
        sub["z_spec_TNS"],
        sub["duration_days"],
        c=sub["SNN_score"],
        cmap="plasma",
        s=60,
        alpha=0.9,
        vmin=0,
        vmax=1,
    )
    plt.colorbar(sc, ax=ax, label="SNN score")

    for _, row in sub.iterrows():
        ax.annotate(
            str(row["TNS_name"])[:12],
            (row["z_spec_TNS"], row["duration_days"]),
            fontsize=6,
            color=MUTED_COL,
            ha="left",
            va="bottom",
        )

    ax.set_xlabel("Spectroscopic redshift z (TNS)", color=MUTED_COL)
    ax.set_ylabel("Observed LC duration (days)", color=MUTED_COL)
    ax.set_title("LC duration vs redshift — TNS SNe", color=ACCENT)
    ax.tick_params(colors=MUTED_COL)
    plt.tight_layout()
    fig.savefig(FIGS_DIR / "duration_vs_redshift.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.show()
else:
    print("Not enough objects with spec-z and duration > 0 to plot the diagnostic.")

## 8 — Individual object deep-dive

Select a specific object by TNS name or diaObjectId to inspect in detail.

In [ ]:
# ── USER SELECTION ────────────────────────────────────────────────────────────
SELECT_BY = "index"  # options: "index", "diaObjectId", "tns_name"
SELECT_VALUE = 0  # 0-based index, integer diaObjectId, or TNS name string
# ─────────────────────────────────────────────────────────────────────────────

if SELECT_BY == "index":
    sel_meta = sorted_catalog.iloc[int(SELECT_VALUE)]
elif SELECT_BY == "diaObjectId":
    rows_ = sorted_catalog[sorted_catalog["r:diaObjectId"] == int(SELECT_VALUE)]
    if rows_.empty:
        raise ValueError(f"diaObjectId {SELECT_VALUE} not found.")
    sel_meta = rows_.iloc[0]
elif SELECT_BY == "tns_name":
    if tns_name_col in sorted_catalog.columns:
        rows_ = sorted_catalog[
            sorted_catalog[tns_name_col].str.contains(str(SELECT_VALUE), na=False, case=False)
        ]
        if rows_.empty:
            raise ValueError(f"TNS name containing '{SELECT_VALUE}' not found.")
        sel_meta = rows_.iloc[0]
    else:
        raise ValueError(f"Column {tns_name_col!r} not in catalog.")

sel_oid = int(sel_meta["r:diaObjectId"])
print(f"Selected object:")
print(f"  diaObjectId  : {sel_oid}")
print(f"  TNS name     : {sel_meta.get(tns_name_col, '—')}")
print(f"  TNS type     : {sel_meta.get(tns_type_col, '—')}")
print(f"  z_spec (TNS) : {sel_meta.get(tns_z_col, 'n/a')}")
print(f"  z_phot DR8   : {sel_meta.get(zphot_col, 'n/a')}")
print(f"  RA / Dec     : {sel_meta['r:ra']:.5f}  /  {sel_meta['r:dec']:.5f}")
print(f"  SNN score    : {sel_meta.get('f:clf_snnSnVsOthers_score', 'n/a')}")
print(f"  earlySNIa    : {sel_meta.get('f:clf_earlySNIa_score', 'n/a')}")
print(f"  SIMBAD otype : {sel_meta.get('f:xm_simbad_otype', '—')}")

In [ ]:
# Load and display the light curve for the selected object
sel_lc = lc_cache.get(sel_oid)
if sel_lc is None:
    lc_p = LC_DIR / f"lc_{sel_oid}.parquet"
    sel_lc = pd.read_parquet(lc_p) if lc_p.exists() else pd.DataFrame()

print(f"{len(sel_lc)} diaSources.  Bands: {sorted(sel_lc['r:band'].unique()) if not sel_lc.empty else []}")
sel_lc.head(10)

In [ ]:
# Full light curve plot for the selected object
plot_tns_lightcurve(sel_lc, sel_meta, save_path=None)

## 9 — Notes on redshift availability

| Column | Provenance | Reliability | Notes |
|--------|-----------|-------------|-------|
| `f:xm_tns_redshift` | TNS spectroscopic report | High | Only filled once a spectrum is submitted to TNS; NaN early in SN evolution |
| `f:xm_legacydr8_zphot` | Legacy Survey DR8 host galaxy photo-z | Medium | Sky coverage ~18 000 deg², mag limit r~23 |
| `f:xm_mangrove_lum_dist` | MANGROVE galaxy catalogue luminosity distance | Medium | Biased toward nearby galaxies |

**Best practice**: use `f:xm_tns_redshift` when available; fall back to `f:xm_legacydr8_zphot` for
distance estimates in the absence of spectroscopy.  
For Hubble diagram work, only spec-z should be used.

**Re-running this notebook**: the light curves are cached in `data_NB07_TNS_SN/light_curves/`.
Re-run cell 4 to refresh from the API (new detections are added nightly for LSST).